# Borrowing time: which injury phases our IR mice carry, and where along the PT each acts (notebook 99)

**The question.** Our IR mice were sampled once, at an unknown time after injury. Kirita 2020's time course (notebook
94's external statistics) defines acute (4–12 h), repair (2 d), late (14 d) and sustained programs from Kirita's mice
alone. Which phases do our IR mice carry, and where along the PT does each act?

**The frozen protocol** is `docs/results/pt-injury-phases.md` (commit 1a92c73).
- A1: each phase program's level in our IR mice against matched random programs; a time-mixture fit.
- A2: for present phases, S3 − S1 location against the same null. H1: acute injury is S3-weighted. H2: repair is
  S3-weighted. H3 (descriptive): location per unit of level by phase.

**Sections.** 1 setup and inputs; 2 phase programs; 3 per-gene effects; 4 A1; 5 A2; 6 figures; 7 reading.

## 1 · Setup and inputs

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'aki' / 'pt_units.h5ad'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
T94 = results_root / 'pt_external_iri' / 'tables'
output = results_root / 'pt_injury_phases'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '99.phases.1'
SEED = 99
FINGERPRINT = 'bb4a80c1b484'
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
SPECIMENS = CONTROLS + IRS
TIMES = ['4hours', '12hours', '2days', '14days', '6weeks']
T_LABEL = {'4hours': '4 h', '12hours': '12 h', '2days': '2 d', '14days': '14 d', '6weeks': '6 wk'}
ALPHA, BIG, QUIET = .05, 1., .5
DETECTION, MIN_MEMBERS = .02, 10
SD_FLOOR, BIN_WIDTH, N_PROFILE_BINS = .25, .05, 20
N_NULL, Z_PRESENT, Z_LOCATED = 1000, 3., 3.
CHUNK = 1000
REQUIRED = [EXPORT, T94 / 'external_genes.csv.gz', T91 / 'i_genes.csv.gz']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 91 and 94 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.optimize import nnls
from IPython.display import display

from pseudospace.external_replication import matched_means
from pseudospace.package_inputs import read_pt_units, support_positions
from pseudospace.pathway_networks import segment_cuts
from pseudospace.tissue_location import gene_effects

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'IR2A2': '#D55E00', 'IR2A4': '#E69F00'}
PHASE_COLOR = {'ACUTE': '#d55e00', 'REPAIR': '#2a78d6', 'LATE': '#9C93D3', 'SUSTAINED': '#1b8a5a'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


guard_log = []
units = read_pt_units(EXPORT, 'aki')
assert units.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, 'G1: fingerprint'
obs = units.obs
mask, position = support_positions(obs.pseudospace.to_numpy(float), obs['sample'].astype(str).to_numpy(),
                                   obs.segment_class, rescale='support')
assert int(mask.sum()) == 14048, 'G1: common support'
SPEC = obs['sample'].astype(str).to_numpy()[mask]
SEG = obs.segment_class.astype(str).str.replace('PT-', '').to_numpy()[mask]
X_POS = position
COUNTS = sparse.csr_matrix(units.X[mask])
GENES = pd.Index(units.var_names.astype(str))
del units
LIB = np.asarray(COUNTS.sum(axis=1)).ravel()
ctrl = np.isin(SPEC, CONTROLS)
C12, C23 = segment_cuts(X_POS[ctrl], SEG[ctrl])
guard_log.append(f'G1: export fingerprint {FINGERPRINT}; 14,048 common-support units; control cuts {C12:.3f}, {C23:.3f}')
print('\n'.join('· ' + g for g in guard_log))

## 2 · Phase programs (from Kirita's data only)

In [ ]:
EXT = pd.read_csv(T94 / 'external_genes.csv.gz', header=[0, 1], index_col=0)
EXT.index = EXT.index.astype(str)
coef = pd.DataFrame({t: EXT[(t, 'level coefficient')] for t in TIMES})
q = pd.DataFrame({t: EXT[(t, 'level q')] for t in TIMES})
DET = {g: np.asarray((COUNTS[np.isin(SPEC, grp)] > 0).mean(axis=0)).ravel() for g, grp in (('ctrl', CONTROLS), ('ir', IRS))}
det_ok = pd.Series(np.all([np.asarray((COUNTS[SPEC == s] > 0).mean(axis=0)).ravel() >= DETECTION for s in CONTROLS], axis=0)
                   | np.all([np.asarray((COUNTS[SPEC == s] > 0).mean(axis=0)).ravel() >= DETECTION for s in IRS], axis=0),
                   index=GENES)
POOL = [g for g in coef.index if g in GENES and det_ok[g]]
print(f"Scored genes: {len(POOL):,} of {len(coef):,} in notebook 94's universe")


def phase(sign):
    big = (q <= ALPHA) & (coef * sign >= BIG)
    quiet = coef * sign < QUIET
    return {
        'ACUTE': (big['4hours'] | big['12hours']) & quiet['2days'] & quiet['14days'],
        'REPAIR': big['2days'] & quiet['4hours'] & quiet['12hours'],
        'LATE': big['14days'] & quiet['4hours'] & quiet['12hours'] & quiet['2days'],
        'SUSTAINED': big['12hours'] & big['2days'] & big['14days'],
    }


PROGRAMS = {}
for sign, tag in ((1, 'up'), (-1, 'down')):
    for name, member in phase(sign).items():
        PROGRAMS[f'{name}-{tag}'] = [g for g in coef.index[member.to_numpy()] if g in set(POOL)]
PROG_TABLE = pd.DataFrame([{'program': k, 'members (scored)': len(v),
                            **{f'mean coefficient {T_LABEL[t]}': float(coef.loc[v, t].mean()) if v else np.nan for t in TIMES}}
                           for k, v in PROGRAMS.items()]).set_index('program')
pd.DataFrame([(k, g) for k, v in PROGRAMS.items() for g in v], columns=['program', 'gene']).to_csv(tables / 'phase_programs.csv', index=False)
display(PROG_TABLE.round(2))
SCORED = [k for k, v in PROGRAMS.items() if len(v) >= MIN_MEMBERS]
print('Scored programs (≥ 10 members):', SCORED)

## 3 · Per-gene injury effects (our units)

In [ ]:
t0 = time.perf_counter()
gpos = GENES.get_indexer(POOL)
edges = np.linspace(0, 1, N_PROFILE_BINS + 1)
pbin = np.clip(np.searchsorted(edges, X_POS, side='right') - 1, 0, N_PROFILE_BINS - 1)
LEVEL, S3S1, PROFILE = {}, {}, {}
for s in SPECIMENS:
    LEVEL[s], S3S1[s], PROFILE[s] = [], [], []
for start in range(0, len(POOL), CHUNK):
    cols = gpos[start:start + CHUNK]
    expr = np.log1p(COUNTS[:, cols].toarray() / LIB[:, None] * 1e4)
    E = gene_effects(expr, ctrl, X_POS, SPEC, CONTROLS, floor=SD_FLOOR, width=BIN_WIDTH)
    for s in SPECIMENS:
        m = SPEC == s
        LEVEL[s].append(E[m].mean(axis=0))
        S3S1[s].append(E[m & (X_POS >= C23)].mean(axis=0) - E[m & (X_POS < C12)].mean(axis=0))
        PROFILE[s].append(np.vstack([E[m & (pbin == b)].mean(axis=0) for b in range(N_PROFILE_BINS)]))
GE = pd.DataFrame({**{f'level {s}': np.concatenate(LEVEL[s]) for s in SPECIMENS},
                   **{f'S3-S1 {s}': np.concatenate(S3S1[s]) for s in SPECIMENS}}, index=pd.Index(POOL, name='gene'))
PROF = {s: np.hstack(PROFILE[s]) for s in SPECIMENS}          # bins x genes
cm = sparse.csr_matrix(COUNTS[ctrl][:, gpos].multiply(1e4 / LIB[ctrl][:, None]))
cm.data = np.log1p(cm.data)
ctrl_expr = np.asarray(cm.mean(axis=0)).ravel()
GE['control expression'] = ctrl_expr
GE['decile'] = np.minimum((GE['control expression'].rank(pct=True) * 10).astype(int), 9)
GE.to_csv(tables / 'gene_effects.csv.gz')
print(f'Gene effects {time.perf_counter() - t0:.0f} s for {len(POOL):,} genes')

## 4 · A1 · Which phases do our IR mice carry?

In [ ]:
stat_cols = [f'level {s}' for s in SPECIMENS] + [f'S3-S1 {s}' for s in SPECIMENS]
pool_index = pd.Series(np.arange(len(POOL)), index=POOL)
ROWS, NULLS = [], {}
for k in SCORED:
    members = PROGRAMS[k]
    obs_vals = GE.loc[members, stat_cols].mean()
    null = matched_means(GE[stat_cols].to_numpy(), GE.decile.to_numpy(), GE.loc[members, 'decile'].to_numpy(), n=N_NULL, seed=SEED)
    NULLS[k] = null
    mu, sd = null.mean(axis=0), null.std(axis=0, ddof=1)
    z = (obs_vals.to_numpy() - mu) / sd
    row = {'program': k, 'members': len(members)}
    for c, v, zz in zip(stat_cols, obs_vals, z):
        row[c] = v
        row[f'z {c}'] = zz
    ROWS.append(row)
A = pd.DataFrame(ROWS).set_index('program')
sign = pd.Series({k: 1. if k.endswith('-up') else -1. for k in A.index})
A['present (version)'] = [(sign[k] * A.loc[k, [f'z level {s}' for s in IRS]] >= Z_PRESENT).all() for k in A.index]
PRESENT = {}
for name in ('ACUTE', 'REPAIR', 'LATE', 'SUSTAINED'):
    versions = [f'{name}-up', f'{name}-down']
    PRESENT[name] = all(v in A.index and A.loc[v, 'present (version)'] for v in versions)
A.to_csv(tables / 'a1_presence.csv')
display(A[[f'z level {s}' for s in SPECIMENS] + ['present (version)']].round(2))
print('Phase present (up and down both, both IR mice):', PRESENT)

common = [g for g in POOL if g in set(pd.read_csv(T91 / 'i_genes.csv.gz').gene.astype(str))]
G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
yv = G91.loc[common, 'L'].to_numpy(float)
Xv = coef.loc[common, TIMES].to_numpy(float)
ok = np.isfinite(yv) & np.isfinite(Xv).all(axis=1)
w, _ = nnls(Xv[ok], yv[ok])
fit = Xv[ok] @ w
r2 = 1 - ((yv[ok] - fit) ** 2).sum() / ((yv[ok] - yv[ok].mean()) ** 2).sum()
MIX = pd.DataFrame({'weight': w, 'share': w / w.sum() if w.sum() > 0 else np.nan}, index=[T_LABEL[t] for t in TIMES])
MIX.loc['R²', 'weight'] = r2
MIX.loc['genes', 'weight'] = int(ok.sum())
MIX.to_csv(tables / 'a1_time_mixture.csv')
display(MIX.round(3))

## 5 · A2 · Where along the PT does each present phase act?

In [ ]:
LOC = []
for k in SCORED:
    name = k.split('-')[0]
    members = PROGRAMS[k]
    prof = {s: PROF[s][:, pool_index[members].to_numpy()].mean(axis=1) for s in SPECIMENS}
    row = {'program': k, 'phase present': PRESENT[name]}
    for s in SPECIMENS:
        row[f'S3-S1 {s}'] = A.loc[k, f'S3-S1 {s}']
        row[f'z S3-S1 {s}'] = A.loc[k, f'z S3-S1 {s}']
        row[f'location per level {s}'] = A.loc[k, f'S3-S1 {s}'] / A.loc[k, f'level {s}'] if abs(A.loc[k, f'level {s}']) > 1e-9 else np.nan
        row[f'peak x {s}'] = float(.5 * (edges[:-1] + edges[1:])[np.nanargmax(sign[k] * prof[s])])
    LOC.append(row)
LOC = pd.DataFrame(LOC).set_index('program')
LOC.to_csv(tables / 'a2_location.csv')
pd.DataFrame({(k, s): PROF[s][:, pool_index[PROGRAMS[k]].to_numpy()].mean(axis=1) for k in SCORED for s in SPECIMENS},
             index=.5 * (edges[:-1] + edges[1:])).to_csv(tables / 'profiles.csv')
display(LOC.round(3))


def located(prog):
    if prog not in LOC.index or not LOC.loc[prog, 'phase present']:
        return None
    return bool((sign[prog] * LOC.loc[prog, [f'z S3-S1 {s}' for s in IRS]] >= Z_LOCATED).all())


VERD = pd.DataFrame([
    {'hypothesis': 'H1 acute injury is S3-weighted', 'up': located('ACUTE-up'), 'down': located('ACUTE-down')},
    {'hypothesis': 'H2 repair is S3-weighted', 'up': located('REPAIR-up'), 'down': located('REPAIR-down')},
])
VERD['holds'] = VERD.apply(lambda r: None if r.up is None or r.down is None else bool(r.up and r.down), axis=1)
VERD.to_csv(tables / 'verdicts.csv', index=False)
display(VERD)

### 5.1 · Post hoc: IR against the control mice's own spread

Added after the run showed that the two control mice, cross-fitted against each other, are not at 0. The A2/A4
factor gives control z up to about ±10 (SUSTAINED-up level), so the matched-random-gene null understates the
between-mouse variation. Here each IR z is set against the larger |z| of the two controls for the same program and
statistic. Not judged.

In [ ]:
CAL = []
for k in SCORED:
    for stat in ('level', 'S3-S1'):
        cz = max(abs(A.loc[k, f'z {stat} {s}']) for s in CONTROLS)
        iz = [A.loc[k, f'z {stat} {s}'] for s in IRS]
        CAL.append({'program': k, 'statistic': stat, 'IR z': ', '.join(f'{v:.2f}' for v in iz), 'max |control z|': cz,
                    'both IR beyond controls (program direction)': bool(all(sign[k] * v > cz for v in iz))})
CAL = pd.DataFrame(CAL)
CAL.to_csv(tables / 'post_hoc_control_spread.csv', index=False)
display(CAL.round(2))

## 6 · Figures

In [ ]:
fig, ax = plt.subplots(figsize=(70 * MM, 50 * MM))
tx = np.arange(len(TIMES))
for k in SCORED:
    name, tag = k.split('-')
    ax.plot(tx, PROG_TABLE.loc[k, [f'mean coefficient {T_LABEL[t]}' for t in TIMES]].to_numpy(float), marker='o', ms=2.5,
            color=PHASE_COLOR[name], ls='-' if tag == 'up' else '--', label=f'{k} ({len(PROGRAMS[k])})')
ax.axhline(0, color=MUTED, lw=.5)
ax.set_xticks(tx, [T_LABEL[t] for t in TIMES])
ax.set_ylabel('mean log2 FC, Kirita IRI')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
save(fig, '99a_phase_programs_kirita')
plt.show()

fig, ax = plt.subplots(figsize=(80 * MM, 50 * MM))
yy = np.arange(len(SCORED))
for j, s in enumerate(SPECIMENS):
    ax.scatter(A.loc[SCORED, f'z level {s}'], yy + (j - 1.5) * .15, s=10, color=SPEC_COLOR[s], label=s,
               marker='o' if s in IRS else 'x')
ax.axvline(0, color=MUTED, lw=.5)
for v in (-Z_PRESENT, Z_PRESENT):
    ax.axvline(v, color=MUTED, lw=.5, ls=':')
ax.set_yticks(yy, SCORED)
ax.invert_yaxis()
ax.set_xlabel('level z against matched random programs')
ax.legend(loc='lower right')
save(fig, '99b_phase_levels')
plt.show()

shown = [k for k in SCORED]
fig, axes = plt.subplots(2, max(1, (len(shown) + 1) // 2), figsize=(180 * MM, 70 * MM), sharex=True, squeeze=False)
centres = .5 * (edges[:-1] + edges[1:])
for ax, k in zip(axes.ravel(), shown):
    for s in SPECIMENS:
        ax.plot(centres, PROF[s][:, pool_index[PROGRAMS[k]].to_numpy()].mean(axis=1), color=SPEC_COLOR[s],
                ls='-' if s in IRS else '--')
    for c in (C12, C23):
        ax.axvline(c, color=MUTED, lw=.5, ls=':')
    ax.axhline(0, color=MUTED, lw=.5)
    ax.set_title(k)
for ax in axes.ravel()[len(shown):]:
    ax.axis('off')
for ax in axes[-1]:
    ax.set_xlabel('PT position')
save(fig, '99c_phase_profiles')
plt.show()

fig, ax = plt.subplots(figsize=(70 * MM, 50 * MM))
for k in SCORED:
    name = k.split('-')[0]
    vals = LOC.loc[k, [f'location per level {s}' for s in IRS]].to_numpy(float)
    ax.scatter([k] * 2, vals, color=PHASE_COLOR[name], s=12)
ax.axhline(0, color=MUTED, lw=.5)
ax.set_ylabel('S3 − S1 per unit of level (IR mice)')
ax.tick_params(axis='x', rotation=45)
save(fig, '99d_location_per_level')
plt.show()

## 7 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', "Kirita's level coefficients are notebook 94's centred whole-PT coefficients; q is notebook 94's BH within each time point."),
    ('reading', 'The time-mixture fit uses no intercept (both sides are centred) over genes in notebook 91 and notebook 94.'),
    ('reading', 'Expression deciles for the matched null: the control mean of log1p(count / library), over the scored genes.'),
    ('reading', 'Profile bins are 20 equal-width bins of x; the peak is the bin of largest signed effect in the program direction.'),
    ('post hoc', '5.1: each IR z against the larger |z| of the two cross-fitted controls (the A2/A4 factor). Not judged.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '99_pt_injury_phases', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'scored genes': len(POOL), 'programs': {k: len(v) for k, v in PROGRAMS.items()}, 'present': PRESENT,
       'time mixture': MIX['share'].dropna().round(3).to_dict(), 'mixture R2': float(r2),
       'verdicts': VERD.to_dict('records'), 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))